In [ ]:
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load GPT model
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Accept prompt from user
prompt = input("Enter your prompt: ")

# Encode input
inputs = tokenizer(prompt, return_tensors="pt")

temperature = 0.7

# Generate response with token scores
output = model.generate(

    **inputs,
    max_new_tokens=20,
    do_sample=True,
    temperature=temperature,
    return_dict_in_generate=True,
    output_scores=True,
)

# Extract only generated tokens (excluding prompt)
generated_tokens = output.sequences[0][inputs["input_ids"].shape[1] :]
generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("\nGenerated Response:")
print(generated_text)

print("\nToken Log Probabilities")
print("-" * 90)
print(
    f"{'Token':15} {'Log Prob':18} {'Probability':15} {'Cumulative Prob':24} {'Color'}"
)
print("-" * 90)

cumulative_probability = 1.0

for token_tensor, score in zip(generated_tokens, output.scores):
    token_id = token_tensor.item()

    # Scale logits by temperature to match generation sampling
    scaled_score = score[0] / temperature
    probabilities = torch.softmax(scaled_score, dim=-1)

    probability = probabilities[token_id].item()
    log_probability = math.log(probability) if probability > 0 else -float("inf")
    cumulative_probability *= probability

    # Color coding
    if probability >= 0.50:
        color = "Green"
    elif probability >= 0.20:
        color = "Yellow"
    else:
        color = "Red"

    # Decode individual token and escape newlines/spaces for clean table alignment
    raw_token = tokenizer.decode([token_id])
    clean_token = repr(raw_token)[1:-1]  # Displays \n and spaces cleanly

    print(
        f"{clean_token:15} {log_probability:<18.6f} {probability:<15.6f} {cumulative_probability:<24.10f} {color}"
    )

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Enter your prompt: education is

Generated Response:
 a critical element in our economy,” he said.




He said the

Token Log Probabilities
------------------------------------------------------------------------------------------
Token           Log Prob           Probability     Cumulative Prob          Color
------------------------------------------------------------------------------------------
 a              -0.497816          0.607857        0.6078565121             Green
 critical       -5.199876          0.005517        0.0033536940             Red
 element        -2.120505          0.119971        0.0004023461             Red
 in             -0.644071          0.525150        0.0002112921             Green
 our            -2.713418          0.066310        0.0000140107             Red
 economy        -2.266762          0.103647        0.0000014522             Red
,               -1.397369          0.247247        0.0000003590             Yellow
�               -1.372934    